1. Importar librerías

In [ ]:
# ==========================================
# 1. Importar librerías
# ==========================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 6)


2. Cargar dataset limpio

In [ ]:
# ==========================================
# 2. Cargar dataset limpio
# ==========================================
import os
project_root = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
file_path = os.path.join(project_root, "data/processed/online_news_cleaned.csv")

df = pd.read_csv(file_path)
df.head()



🟦 Grupo A — Variables de Tokens (Texto)

Variables típicas del grupo:

In [ ]:
token_vars = [
    "n_tokens_title",
    "n_tokens_content",
    "n_unique_tokens",
    "n_non_stop_words",
    "n_non_stop_unique_tokens",
    "average_token_length"
]


🔵 1. Revisión Rápida de Estadísticas
Te permite ver:
medias
mínimos
máximos
rango
desviaciones

In [ ]:
df[token_vars].describe().T


🔵 2. Histogramas + KDE (para ver distribuciones)

Objetivo:
Detectar si la variable está sesgada (skewed)
Detectar si necesita transformación (log)
Ver outliers

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

for col in token_vars:
    plt.figure(figsize=(8,4))
    sns.histplot(df[col], kde=True, bins=40)
    plt.title(f"Distribución de {col}")
    plt.show()


🔵 3. Boxplots (detección visual de outliers)

Muy útil para:
Distorsión en escalas
Rango real de la variable
Detección de puntos extremos

In [ ]:
for col in token_vars:
    plt.figure(figsize=(8,2))
    sns.boxplot(x=df[col])
    plt.title(f"Boxplot de {col}")
    plt.show()


🔵 4. Correlación individual con la variable objetivo

(“shares” o “log_shares” recomendado)

Esto sirve para entender:
Si la longitud del texto influye
Qué tan fuerte es la relación

In [ ]:
(df[token_vars + ["shares"]].corr()["shares"]
    .sort_values(ascending=False))


In [ ]:
# Crear variable transformada
df["log_shares"] = np.log1p(df["shares"])

# Correlación con variables del grupo A
(
    df[token_vars + ["log_shares"]]
    .corr()["log_shares"]
    .sort_values(ascending=False)
)


🔵 5. Pairplot (relaciones cruzadas dentro del grupo A)

Esto detecta patrones y dependencias internas:

In [ ]:
sns.pairplot(df[token_vars], diag_kind="kde")
plt.show()


🔵 6. Análisis bivariado: cada token vs shares

Scatterplots para ver relaciones no lineales:

In [ ]:
for col in token_vars:
    plt.figure(figsize=(6,4))
    sns.scatterplot(x=df[col], y=df["shares"])
    plt.title(f"{col} vs shares")
    plt.show()


In [ ]:
for col in token_vars:
    plt.figure(figsize=(6,4))
    sns.scatterplot(x=df[col], y=df["log_shares"])
    plt.title(f"{col} vs log_shares")
    plt.show()


🔵 7. Normalización recomendada (opcional para el análisis)

Si ves que la variable está muy sesgada:

In [ ]:
import numpy as np

df["n_tokens_content_log"] = np.log1p(df["n_tokens_content"])


In [ ]:
sns.histplot(df["n_tokens_content_log"], kde=True)
plt.show()


🔵 8. Evaluar multicolinealidad dentro del grupo

(VIF solo para este grupo)

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
import pandas as pd

X = df[token_vars].dropna()
vif_data = pd.DataFrame()
vif_data["feature"] = X.columns
vif_data["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
vif_data


# 🟦 Conclusiones del Grupo A — Variables de Tokens (Texto)

El análisis del Grupo A incluye las variables:

- `n_tokens_title`
- `n_tokens_content`
- `n_unique_tokens`
- `n_non_stop_words`
- `n_non_stop_unique_tokens`
- `average_token_length`

Estas variables representan la **longitud del texto**, **riqueza léxica**, **complejidad lingüística** y **variedad de vocabulario** de los artículos.

---

## 📌 1. Distribuciones sesgadas y gran variabilidad en la longitud del contenido

- `n_tokens_content` muestra una distribución **altamente sesgada**, con artículos muy cortos (0–200 tokens) y artículos extremadamente largos (hasta ~2640 tokens).  
- `n_tokens_title` tiene valores más estables, centrados en 9–12 tokens, pero con *outliers extremos* (hasta 124 tokens).  
- Las métricas normalizadas (`n_unique_tokens`, `n_non_stop_words`, etc.) están concentradas entre 0.5 y 1.0, mostrando poca variabilidad.

👉 **Implicación:** algunas variables necesitan *transformación logarítmica* o *escalamiento*, especialmente `n_tokens_content`.

---

## 📌 2. Outliers significativos en varias variables

Los boxplots muestran valores extremos en:

- `n_tokens_content`
- `n_unique_tokens` (algunos artículos tienen 0 o 1.0)
- `n_non_stop_unique_tokens`
- `average_token_length` (valores extrañamente altos de hasta 76)

👉 **Implicación:**  
Los outliers deben ser gestionados antes del modelado, ya sea mediante:
- winsorización,
- log-transformación,
- o normalización robusta.

---

## 📌 3. Correlación muy débil con la variable objetivo (`shares` y `log_shares`)

Correlaciones con `shares`:

- Entre -0.03 y -0.005  
- Ninguna variable supera |0.03| de correlación.

Correlaciones con `log_shares`:

- Entre -0.056 y 0.015  
- Ninguna es predictiva por sí sola.

👉 **Conclusión:**  
**La longitud o complejidad del texto no tiene relación lineal directa con la viralidad.**  
Esto es consistente con estudios previos: *el tamaño del artículo rara vez determina los shares*.

---

## 📌 4. Relaciones internas fuertes entre variables léxicas

El pairplot y el VIF muestran que:

- `n_unique_tokens`
- `n_non_stop_words`
- `n_non_stop_unique_tokens`

están fuertemente correlacionadas entre sí.

👉 **Con VIFs extremadamente altos (67–94)** queda claro que estas variables son **altamente redundantes** y capturan la misma información:

- riqueza léxica  
- proporción de palabras únicas  
- proporción de palabras no-stop  

👉 **Implicación:**  
En Feature Engineering se recomienda:

- Seleccionar solo **una** o **dos** métricas representativas.  
- Eliminar las redundantes para evitar multicolinealidad y mejorar modelos lineales.

---

## 📌 5. Análisis bivariado confirma relaciones no lineales y dispersas

Las gráficas de dispersión (`tokens vs shares/log_shares`) muestran:

- Nubes de puntos sin patrón.
- Ausencia de tendencia lineal o no lineal clara.

👉 **Esto indica que estas variables solo aportarán valor en modelos de interacción o árboles**, no en regresión lineal.

---

## 📌 6. Recomendación de transformaciones

Debido al sesgo y a los outliers:

- `n_tokens_content` → aplicar `np.log1p()`  
- `average_token_length` → revisar valores anómalos  
- variables proporcionales (`n_unique_tokens`, etc.) → escalar con MinMaxScaler o RobustScaler

Esto ayudará a estabilizar la varianza y mejorar el entrenamiento de algunos modelos.

---

## 📌 7. Conclusión general del Grupo A

El análisis revela que:

### ✔ La longitud del artículo es muy variable y muestra outliers  
### ✔ No existe relación directa entre longitud y viralidad  
### ✔ Las métricas léxicas están altamente correlacionadas entre sí (multicolinealidad fuerte)  
### ✔ Se deben seleccionar variables representativas y eliminar redundantes  
### ✔ Transformaciones y escalamiento serán necesarios en Feature Engineering  

En resumen:

> **Las variables de tokens no predicen por sí solas la viralidad, pero aportan información estructural útil cuando se integran con otros grupos de características.**

---

¿Deseas ahora el bloque de código en Markdown para el **Feature Engineering del Grupo A**, o seguimos con el **Grupo C — Keywords (`kw_*`)**, que es el grupo más importante del dataset?


# ✅ Conclusiones Finales del Grupo A (Tokens)  
## ➡️ Requerimientos para el Pipeline de Preprocesamiento en MLOps

Basado en el análisis completo del Grupo A, estas son las conclusiones enfocadas 100% en lo que tu **pipeline de scikit-learn** debe contemplar.

---

## 🔹 1. Necesidad de transformar variables sesgadas (log1p)

Varias variables presentan distribuciones altamente sesgadas y con colas largas:

- `n_tokens_content`  
- `average_token_length`  
- Algunas métricas proporcionales (`n_unique_tokens`, `n_non_stop_unique_tokens`, etc.)  

**→ Acción para el pipeline:**  
Incluir un **Pipeline de transformación logarítmica** usando:

```python
FunctionTransformer(np.log1p)

## 🔹 2. Variables redundantes → riesgo de multicolinealidad

El VIF mostró valores extremadamente altos (67–94) para:

- `n_unique_tokens`
- `n_non_stop_words`
- `n_non_stop_unique_tokens`

Estas variables capturan esencialmente la misma información.

**→ Acción para el pipeline:**

- Seleccionar solo **1 o 2** de estas métricas.  
- O dejar que modelos no lineales manejen la redundancia, pero asegurando **escalado obligatorio** para evitar explosión numérica.

---

## 🔹 3. Requieren escalado para evitar que dominen el modelo

Las variables numéricas del Grupo A tienen magnitudes muy distintas:

- `n_tokens_content` puede ir de **0 a 2640**
- Proporciones entre **0 y 1**
- Outliers en `average_token_length` hasta **76**

**→ Acción para el pipeline:**  
Crear un pipeline de escalado usando:

- `MinMaxScaler` para modelos basados en distancias o redes neuronales  
- `RobustScaler` si mantienes outliers extremos

---

## 🔹 4. Correlación débil → no aportan mucho por sí solas

Las correlaciones con `shares` y `log_shares` son prácticamente nulas.  
Aun así, aportan estructura combinadas con otros grupos de features.

**→ Acción para el pipeline:**

- Incluirlas en el **ColumnTransformer**, pero no esperar alto poder predictivo individual.

---

## 🔹 5. No requieren OneHotEncoder

Las variables del Grupo A son **numéricas**, no categóricas.

**→ Acción para el pipeline:**

- Enviarlas únicamente a:
  - Pipeline de transformación log  
  - Pipeline de escalado  
- **No** deben pasar por OHE.

---

## 🔹 6. Manejo de outliers

Los boxplots mostraron valores muy fuera de escala:

- Hasta **2640 tokens** en contenido  
- Hasta **76** en longitud promedio  

**→ Acción para el pipeline:**

- Usar `np.log1p` para reducir el impacto de valores extremos  
- Alternativa: `RobustScaler` si decides no transformar con log

---

## 🔹 7. Agrupar estas variables en el ColumnTransformer

Todas las variables del Grupo A deben agruparse en una lista clara:

```python
token_vars = [
    "n_tokens_title",
    "n_tokens_content",
    "n_unique_tokens",
    "n_non_stop_words",
    "n_non_stop_unique_tokens",
    "average_token_length"
]



# ==========================================
# 1. Importar librerías
# ==========================================
import numpy as np
import pandas as pd

from sklearn.preprocessing import (
    FunctionTransformer, MinMaxScaler, RobustScaler,
    OneHotEncoder
)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer


# ==========================================
# 2. Definir las listas de columnas
# ==========================================

# Grupo A: Tokens
token_vars = [
    "n_tokens_title",
    "n_tokens_content",
    "n_unique_tokens",
    "n_non_stop_words",
    "n_non_stop_unique_tokens",
    "average_token_length"
]

# Variables a transformar con log (solo las con skew fuerte)
token_vars_for_log = [
    "n_tokens_content",
    "average_token_length"
]

# Variables redundantes o proporcionales → solo escalar
token_vars_for_scaling = [
    "n_tokens_title",
    "n_unique_tokens",
    "n_non_stop_words",
    "n_non_stop_unique_tokens"
]

# Detectar categóricas automáticamente
categorical_vars = [col for col in df.columns if df[col].dtype == "object"]

# Detectar numéricas (restando las categóricas)
numeric_vars = [col for col in df.columns 
                if col not in categorical_vars + ["shares"]]


# ==========================================
# 3. Pipelines individuales
# ==========================================

# 🔹 Pipeline para variables log-transform
log_pipeline = Pipeline(steps=[
    ("log_transform", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
    ("scaler", MinMaxScaler())
])

# 🔹 Pipeline para variables solo con escalado
scale_pipeline = Pipeline(steps=[
    ("scaler", MinMaxScaler())  # O RobustScaler()
])

# 🔹 Pipeline para variables categóricas
categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

# 🔹 Pipeline para variables numéricas restantes
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", MinMaxScaler())
])


# ==========================================
# 4. ColumnTransformer – ESTRUCTURA CENTRAL DEL PIPELINE
# ==========================================

preprocessor = ColumnTransformer(
    transformers=[
        ("token_log", log_pipeline, token_vars_for_log),
        ("token_scale", scale_pipeline, token_vars_for_scaling),
        ("categorical", categorical_pipeline, categorical_vars),
        ("numeric", numeric_pipeline, numeric_vars)
    ],
    remainder="drop"
)


# ==========================================
# 5. Pipeline final listo para entrenamiento
# ==========================================

pipeline_final = Pipeline(steps=[
    ("preprocessing", preprocessor)
    # Aquí conecta tu modelo:
    # ("regressor", RandomForestRegressor())
])

pipeline_final
